# Strict Reproduction — Retained Spectral credibility audit

This notebook installs one **immutable audited commit** and one frozen CPU dependency set, then runs the measured benchmark from scratch. It does not load the repository's committed result JSON.

The strict audit requires both the native RMS pipeline and an independent SciPy pipeline to hit each declared tolerance and return `ACCEPT`; timing order is randomized; the requested-only SciPy tridiagonal comparator must cross-check; eight adversarial cases and a 20-point `(grid size, requested modes)` scaling sweep are also run.

Scope: `finite_diagnostic` one-dimensional spectra. The measured claim is limited to the declared suite. The scaling sweep is reported explicitly and is **not** a universal speed-dominance claim.

In [ ]:
AUDITED_COMMIT = '3dbe6c93a5841f24fd3d055948e8ae4ccd4ff877'
!pip -q install numpy==1.26.4 scipy==1.13.1 numba==0.60.0 llvmlite==0.43.0 matplotlib==3.9.2 jax==0.4.35 jaxlib==0.4.35
!pip -q install "information-discrete-math @ git+https://github.com/morrocwi/information-discrete-math@{AUDITED_COMMIT}"
print('audited source commit:', AUDITED_COMMIT)

In [ ]:
import os
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
os.environ['PYTHONHASHSEED'] = '0'
os.environ['JAX_PLATFORMS'] = 'cpu'
# The audit records a source identifier even outside GitHub Actions.
os.environ['GITHUB_SHA'] = AUDITED_COMMIT

In [ ]:
import json
from pathlib import Path
from retained_spectral.competition.credibility_audit import run_credibility_audit

result = run_credibility_audit(
    include_jax=True,
    baseline_repeats=9,
    executor_repeats=5,
    scaling_repeats=7,
)
Path('credibility-audit.json').write_text(json.dumps(result, indent=2, sort_keys=True))
print('overall verdict:', result['verdict'])
print('strict gates:', json.dumps(result['gates'], indent=2))
print('baseline:', json.dumps(result['baseline']['end_to_end']['summary'], indent=2))
scaling = list(result['scaling']['cases'].values())
print('scaling native wins:', sum(c['scipy_to_native_time_ratio'] > 1 for c in scaling), '/', len(scaling))

In [ ]:
from pathlib import Path
from retained_spectral.competition.chart import render_hero, render_detail
from IPython.display import Image, display

# Existing charts consume the strict baseline record nested inside the audit.
render_hero(result['baseline'], Path('strict-hero.png'))
render_detail(result['baseline'], Path('strict-detail.png'))
display(Image('strict-hero.png'))
display(Image('strict-detail.png'))

In [ ]:
# Inspect the four scaling points where SciPy may be faster on this host.
slower = {
    name: case for name, case in result['scaling']['cases'].items()
    if case['scipy_to_native_time_ratio'] <= 1.0
}
print(json.dumps(slower, indent=2, sort_keys=True))

In [ ]:
# Solve a new raw-input problem after the audit.
import retained_spectral as rs
problem = rs.make_problem(
    name='my', family='harmonic',
    parameters={'omega': 2.0, 'center': 0.0}, modes=4
)
readout = rs.solve(problem)
print(readout.status, readout.values)